# LLM Uncertainty Benchmark — NVIDIA NIM API

Runs the full CP benchmark using **NVIDIA NIM** (hosted inference API).  
No local GPU needed — models run on NVIDIA's servers.

**Advantages over Ollama:**
- Real token logprobs from a single API call (not continuation scoring approximation)
- Access to large models (70B+) without local hardware
- Much faster — parallel inference on NIM servers

**Setup:** Get a free API key at https://build.nvidia.com → top right → "Get API Key"

## 1. Install dependencies and clone repo

In [ ]:
!pip install -q numpy scikit-learn tqdm requests matplotlib

import subprocess, os, sys
if not os.path.exists('LLM-Uncertainty-Study'):
    subprocess.run(['git', 'clone', 'https://github.com/SokhengDin/LLM-Uncertainty-Study.git'],
                   check=True)
os.chdir('LLM-Uncertainty-Study')
print('Working directory:', os.getcwd())

## 2. Configuration — set your API key and models here

In [ ]:
import os

# ── Your NVIDIA API key ───────────────────────────────────────────────────────
# Get one free at https://build.nvidia.com (top right → "Get API Key")
NVIDIA_API_KEY = "nvapi-xxxxxxxxxxxxxxxxxxxx"   # <-- paste your key here
NIM_BASE_URL   = "https://integrate.api.nvidia.com/v1"

# ── Models to benchmark ───────────────────────────────────────────────────────
# Full NIM model IDs — see utils/nim_client.py for shorthand aliases
MODELS = [
    "meta/llama-3.1-8b-instruct",
    "meta/llama-3.1-70b-instruct",
    "qwen/qwen2.5-7b-instruct",
    # "qwen/qwen2.5-72b-instruct",   # uncomment for larger model
    # "google/gemma-2-9b-it",
]

# ── Tasks ─────────────────────────────────────────────────────────────────────
DATASETS = [
    'mmlu_10k',
    'cosmosqa_10k',
    'hellaswag_10k',
    'halu_dialogue',
    'halu_summarization',
]

# ── CP settings ───────────────────────────────────────────────────────────────
# n=100 → 10 few-shot reserved + 100 test = 110 loaded
# After removing 5 few-shot demonstration indices: 105 items
# 50% cal split → n_cal=52, qhat = ceil(53*0.9)/52 = 48/52 = 0.923  (non-trivial!)
SAMPLES = 100
PROMPT  = 'base'
ICL     = 'icl1'
ALPHA   = 0.1

os.makedirs('outputs_nim', exist_ok=True)
os.makedirs('figures_nim', exist_ok=True)
print(f'Config: {len(MODELS)} models × {len(DATASETS)} tasks × n={SAMPLES}')
print(f'Expected n_cal ≈ 52 → qhat ≈ 0.923 (non-trivial CP)')

## 3. Quick API test — verify key and logprobs work

In [ ]:
import sys
sys.path.insert(0, 'utils')
from nim_client import NIMClient
import numpy as np

client = NIMClient(model_name=MODELS[0], api_key=NVIDIA_API_KEY, base_url=NIM_BASE_URL)

test_prompt = """Question: What is the capital of France?
Choices:
A. Berlin
B. Madrid
C. Paris
D. Rome
E. I don't know
F. None of the above
Answer:"""

logits = client.get_choice_logits(test_prompt)
probs  = np.exp(logits) / np.exp(logits).sum()
choices = ['A', 'B', 'C', 'D', 'E', 'F']

print(f'Model: {MODELS[0]}')
print('Choice logprobs:')
for c, lp, p in zip(choices, logits, probs):
    bar = '█' * int(p * 30)
    print(f'  {c}: {lp:7.3f}  ({p:.3f}) {bar}')
print(f'\nPredicted: {choices[np.argmax(logits)]}  (correct: C)')

## 4. Generate logits for all models × tasks

In [ ]:
import json, pickle, random
from tqdm.notebook import tqdm
sys.path.insert(0, 'utils')
import prompt as pt
from nim_client import NIMClient

FEW_SHOT_IDS = {
    "MMLU"               : [1, 3, 5, 7, 9],
    "HellaSwag"          : [1, 3, 5, 7, 9],
    "CosmosQA"           : [1, 3, 5, 7, 9],
    "Halu-OpenDialKG"    : [5, 7, 9],
    "Halu-CNN/DailyMail" : [9],
}
FEW_SHOT_RESERVE = 10


def load_data(data_file, max_samples):
    data = json.load(open(data_file))
    few_shot_data  = data[:FEW_SHOT_RESERVE]
    remaining_data = data[FEW_SHOT_RESERVE:]
    if len(remaining_data) > max_samples:
        random.seed(42)
        remaining_data = random.sample(remaining_data, max_samples)
    data = few_shot_data + remaining_data
    print(f"  Loaded: {FEW_SHOT_RESERVE} few-shot + {len(remaining_data)} test = {len(data)} total")
    return data


def get_fewshot_exps(data):
    src = data[0]["source"]
    return [data[idx] for idx in FEW_SHOT_IDS[src]]


def format_example(example, prompt, with_answer=False):
    src = example["source"]
    if src == "MMLU":
        prompt += "Question: " + example["question"] + "\nChoices:\n"
    elif src in ("CosmosQA", "HellaSwag"):
        prompt += "Context: "  + example["context"]  + "\n"
        prompt += "Question: " + example["question"] + "\nChoices:\n"
    elif src == "Halu-OpenDialKG":
        prompt += "Dialogue: " + example["context"]  + "\n"
        prompt += "Question: " + example["question"] + "\nChoices:\n"
    elif src == "Halu-CNN/DailyMail":
        prompt += "Document: " + example["context"]  + "\n"
        prompt += "Question: " + example["question"] + "\nChoices:\n"
    for k, v in example["choices"].items():
        prompt += k + ". " + str(v) + "\n"
    prompt += "Answer:"
    if with_answer:
        prompt += " " + example["answer"] + "\n"
    return prompt


def format_base_prompt(example, fewshot_exps):
    prompt = ""
    for exp in fewshot_exps:
        prompt = format_example(exp, prompt, with_answer=True)
    return {"id": example["id"], "prompt": format_example(example, prompt)}


for model_id in MODELS:
    short = model_id.split("/")[-1]
    client = NIMClient(model_name=model_id, api_key=NVIDIA_API_KEY, base_url=NIM_BASE_URL)

    for dataset in DATASETS:
        out_path = f'outputs_nim/{short}_{dataset}_base_icl1_sample{SAMPLES}.pkl'
        if os.path.exists(out_path):
            print(f'  SKIP {short} | {dataset} (already done)')
            continue

        print(f'\n--- {short} | {dataset} ---')
        data         = load_data(f'data/{dataset}.json', SAMPLES)
        fewshot_exps = get_fewshot_exps(data)
        prompt_data  = [format_base_prompt(d, fewshot_exps) for d in data]

        outputs = []
        for exp in tqdm(prompt_data, desc=f'{short}|{dataset}'):
            logits = client.get_choice_logits(exp["prompt"])
            outputs.append({"id": exp["id"], "logits_options": logits})

        with open(out_path, 'wb') as f:
            pickle.dump(outputs, f)
        print(f'  Saved → {out_path}')

print('\nAll logits generated!')

## 5. Run CP evaluation

In [ ]:
import subprocess, sys

for model_id in MODELS:
    short = model_id.split("/")[-1]
    print(f'\n=== Evaluating {short} ===')
    ret = subprocess.run([
        sys.executable, 'main.py',
        '--model',          short,
        '--data_names',     *DATASETS,
        '--prompt_methods', PROMPT,
        '--icl_methods',    ICL,
        '--max_samples',    str(SAMPLES),
        '--alpha',          str(ALPHA),
        '--logits_data_dir', 'outputs_nim',
        '--output_dir',      'outputs_nim',
    ], capture_output=False)
    if ret.returncode != 0:
        print(f'ERROR evaluating {short}')

## 6. Generate figures

In [ ]:
ret = subprocess.run([
    sys.executable, 'plot_results.py',
    '--samples',    str(SAMPLES),
    '--prompt',     PROMPT,
    '--icl',        ICL,
    '--results_dir', 'outputs_nim',
    '--figures_dir', 'figures_nim',
])
print('Figures saved to figures_nim/')

## 7. Display figures

In [ ]:
from IPython.display import Image, display
import glob

for png in sorted(glob.glob('figures_nim/*.png')):
    print(f'\n{png}')
    display(Image(png))

## 8. Summary table

In [ ]:
import json, numpy as np

key = f'{PROMPT}_{ICL}'
col = 16

header = f"{'Model':<30}" + ''.join(f"{d.split('_')[0]:>{col}}" for d in DATASETS)
print(header)
print(f"{'':30}" + ''.join(f"{'CR%/Acc%/SS':>{col}}" for _ in DATASETS))
print('-' * (30 + col * len(DATASETS)))

for model_id in MODELS:
    short = model_id.split('/')[-1]
    path  = f'outputs_nim/{short}_all_results.json'
    if not os.path.exists(path):
        print(f'{short:<30}  (no results)')
        continue
    res = json.load(open(path))
    row = f'{short:<30}'
    for d in DATASETS:
        if d not in res or key not in res[d].get('Acc', {}):
            row += f"{'N/A':>{col}}"
            continue
        acc = 100 * res[d]['Acc'][key]
        cr  = 100 * np.mean([res[d]['LAC_coverage'][key], res[d]['APS_coverage'][key]])
        ss  =       np.mean([res[d]['LAC_set_size'][key],  res[d]['APS_set_size'][key]])
        cell = f"{cr:.0f}/{acc:.0f}/{ss:.1f}"
        row += f"{cell:>{col}}"
    print(row)

print(f'\nn={SAMPLES} → n_cal≈52 → qhat≈0.923 (non-trivial CP, α={ALPHA})')

## 9. Download results

In [ ]:
!zip -r nim_results.zip outputs_nim/ figures_nim/
try:
    from google.colab import files
    files.download('nim_results.zip')
except ImportError:
    print('Not on Colab — results saved to nim_results.zip')